In [25]:
import yaml
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
import re
import numpy as np
import matplotlib.pyplot as plt
from typing import Any, Dict, List, Optional, Tuple, Union
import pandas as pd

#from typing import Any, Dict, List, Union

from tqdm import tqdm
import json

from thesis_project.models.keyword_spotting import KWSBase, KWSDynamic
from thesis_project.models.components.routers import GRURouter
from thesis_project.utils.paths import get_data_dir
from thesis_project.datasets import SpeechCommandsGoogle
from base_config_temp import cfg, noise_train_cfg, noise_eval_cfg

# Device selection: CUDA > MPS > CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
    pin_memory = True
    num_workers = 16
    print("Using CUDA")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    pin_memory = False
    num_workers = 0
    print("Using MPS")
else:
    device = torch.device("cpu")
    pin_memory = False
    print("Using CPU")

Using MPS


In [26]:
def router_acceptable_avg_rank(
    GRU,
    pw_in: float,
    pw_out: float,
    num_pw_layers: int,
    best_static_model_rank: float,
    T: int,
    include_fc_once: bool = False,
) -> float:
    """
    Break-even average rank for a router that:
      - runs a GRU over all T frames (same time length as the PW layers)
      - then applies a final Linear(H -> 1) head ONCE to output a scalar rank

    Uses your symbolic MAC model:
      GRU per-frame MACs: 3H(R_in + H) + (L-1)*6H^2
      FC-once MACs:       H            (Linear(H->1))
      PW per-frame MACs:  num_pw_layers * (pw_in + pw_out)

    Break-even condition (dynamic <= static):
      T*GRU_pf + H + num_pw_layers*T*r*(pw_in+pw_out) <= num_pw_layers*T*r_static*(pw_in+pw_out)

    Returns max(1, r_break_even).
    """
    if T <= 0:
        raise ValueError(f"T must be > 0, got {T}")
    if num_pw_layers <= 0:
        raise ValueError(f"num_pw_layers must be > 0, got {num_pw_layers}")

    H = float(GRU.hidden_size)
    R_in = float(GRU.input_size)
    L = int(GRU.num_layers)

    A = float(pw_in) + float(pw_out)
    if A <= 0:
        raise ValueError(f"(pw_in + pw_out) must be > 0, got {A}")

    pw_macs_per_frame = num_pw_layers * A
    gru_macs_per_frame = 3.0 * H * (R_in + H) + (L - 1) * 6.0 * (H ** 2)
    # Final head: Linear(H -> 1) run once
    fc_once_macs = H if include_fc_once else 0

    break_even_rank = (
        best_static_model_rank
        - (gru_macs_per_frame / pw_macs_per_frame)
        - (fc_once_macs / (T * pw_macs_per_frame))
    )

    acceptable_avg_rank = max(1.0, float(break_even_rank))

    print(
        f"GRU MACs/frame: {gru_macs_per_frame:.2f}, "
        f"FC MACs once: {fc_once_macs:.2f}, "
        f"PW MACs/frame: {pw_macs_per_frame:.2f}, "
        f"Break-even rank: {break_even_rank:.3f}, "
        f"Acceptable avg rank: {acceptable_avg_rank:.3f}"
    )
    return acceptable_avg_rank




In [35]:
maximum_useful_rank = 64
# Model
base_model = KWSBase(cfg).to(device)
#base_model.load_state_dict(torch.load("model_runs/base/2025-12-14_18-48-50/best_model.pth"))
# check base model input dim to set router input dim

gru_in = base_model.frontend.out_channels

router = GRURouter(
    input_dim=gru_in,
    gru_hidden_dim=48,
    num_gru_layers=1,
    max_rank=maximum_useful_rank,
    last_layer_bias_init= None # to bias towards full rank at start
    ).to(device)

# check acceptable average rank
T_example = 63  # example time frames after frontend for 1s input
pw_in = base_model.frontend.out_channels
pw_out = base_model.frontend.out_channels
num_pw_layers = len(base_model.backbone) * 3 * 2 # each stack has 3 bloack, and one block has 2 CPC_Conv1d layers

acceptable_avg_rank = router_acceptable_avg_rank(
    GRU=router.gru,
    pw_in=pw_in,
    pw_out=pw_out,
    num_pw_layers=num_pw_layers,
    best_static_model_rank=30.0,  # from previous experiments
    T=T_example,
    include_fc_once=True,
)


GRU MACs/frame: 25344.00, FC MACs once: 48.00, PW MACs/frame: 4608.00, Break-even rank: 24.500, Acceptable avg rank: 24.500
